In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score,f1_score,confusion_matrix

week7_record = {
    "params": {"max_depth": 3, "min_samples_leaf": 1, "class_weight": "balanced"},
    "random_state": 42, "threshold": 0.5,
    "test_recall": 0.745, "test_f1": 0.536,
    "found": 41, "missed": 14, "false_alarms": 57,
}
df = pd.read_csv(r"C:\Users\ADMIN\OneDrive\Desktop\5thsem_OJT\project1_diabetes\data\processed\dilna_P1W1_features.csv")
df=pd.read_csv(r'C:\Users\ADMIN\OneDrive\Desktop\5thsem_OJT\project1_diabetes\data\processed\dilna_P1W1_features.csv')
feature_cols=['age','pregnancies','glucose','blood_pressure','skin_thickness',
'insulin','bmi','diabetes_pedigree','bmi category','glucose category']



x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]

x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)

final_model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    max_depth=3,
    min_samples_leaf=1
)

final_model.fit(x_train, y_train)
print("Final Random Forest trained.")

pred = final_model.predict(x_test)

proba = final_model.predict_proba(x_test)[:, 1]

cm = confusion_matrix(y_test, pred)
tn, fp, fn, tp = cm.ravel()

test_recall = recall_score(y_test, pred, zero_division=0)
test_f1 = f1_score(y_test, pred, zero_division=0)


print("Test recall:", round(test_recall, 3))
print("Test F1:", round(test_f1, 3))

print(f"found {tp} | missed {fn} | false alarms {fp}")

Final Random Forest trained.
Test recall: 0.745
Test F1: 0.536
found 41 | missed 14 | false alarms 57


# what does the model rely on , and does it make medicals sense

Feature Importance: Identifies which features the model considers important.

Permutation Importance: Measures how much model performance decreases when a feature is shuffled.

In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(final_model, x_test, y_test, scoring = "recall")
                           
both = pd.DataFrame({"built_in": final_model.feature_importances_,
                     "perm_mean": pi.importances_mean,
                     "perm_std": pi.importances_std}, index=x.columns)
both["rank_built_in"] = both["built_in"].rank(ascending=False).astype(int)
both["rank_perm"]     =both["perm_mean"].rank(ascending=False).astype(int)
print(both.sort_values("perm_mean",ascending=False).round(3).to_string())

print("\nshare of build-in importance held by to three :",
        round (both["built_in"].nlargest(3).sum(),3))

                              built_in  perm_mean  perm_std  rank_built_in  rank_perm
glucose                          0.323      0.196     0.040              1          1
glucose category_normal          0.148      0.116     0.036              2          2
insulin                          0.047      0.033     0.018              6          3
bmi category_over weight         0.011      0.029     0.009             11          4
age                              0.135      0.018     0.023              3          5
blood_pressure                   0.031      0.018     0.000              8          5
diabetes_pedigree                0.099      0.015     0.039              5          7
bmi category_obese               0.024      0.015     0.018              9          7
bmi                              0.111      0.007     0.019              4          9
skin_thickness                   0.036      0.004     0.007              7         10
pregnancies                      0.023      0.000     

## count the call-backs - the threshold as a capacity dial

In [ ]:
print(f"Flagged {pred.sum()} of {len(pred)}="
       f"({pred.mean() * 100:.1f}%) patients screened")
yt = y_test.values
rows = []

for t in [0.40, 0.45, 0.50, 0.55, 0.60]:
  p = (proba >= t).astype(int)

  rows.append({
        'threshold': t, 
        'flagged_per_100': round(p.mean() * 100, 1), 
        'found': int(((p == 1) & (y_test == 1)).sum()), 
        'missed': int(((p == 0) & (y_test == 1)).sum()), 
        'healthy_called_back': int(((p == 1) & (y_test == 0)).sum())})

menu = pd.DataFrame(rows)
print(menu.to_string(index=False))

mid = (proba >= 0.49) & (proba < 0.60)
print(f"\nscore between 0.40 and 0.60: {mid.sum()} of {len(proba)}patients "
f"({mid.mean():.0%}): diabetic among them: {yt[mid].mean():.0%} "
f"(overall: {yt.mean():.0%})")

Flagged 98 of 190=(51.6%) patients screened
 threshold  flagged_per_100  found  missed  healthy_called_back
      0.40             73.7     52       3                   88
      0.45             65.3     48       7                   76
      0.50             51.6     41      14                   57
      0.55             30.0     27      28                   30
      0.60             11.1     13      42                    8

score between 0.40 and 0.60: 80 of 190patients (42%): diabetic among them: 35% (overall: 29%)


In [ ]:
test_df = df.loc[x_test.index].copy()
test_df["flagged"]= pred

for col in ["age band","bmi category","glucose category"]:
    rows = []
    for value, sub in test_df.groupby(col):
        pos = int((sub["outcome"] == 1).sum())
        hit = int(((sub["outcome"] == 1) & (sub["flagged"] == 1)).sum())
        rows.append({col: value, "diabetic": pos, "found": hit,
                     "missed": pos - hit,
                     "recall": round(hit / pos, 2) if pos else None})
print(pd.DataFrame(rows).to_string(index=False), "\n")

glucose category  diabetic  found  missed  recall
        diabetic        30     28       2    0.93
          normal         4      0       4    0.00
     prediabetic        21     13       8    0.62 



In [ ]:
df

,Unnamed: 0,patient_id,age,gender,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,physical_activity,smoker,last_checkup_date,outcome,age band,bmi category,glucose category,insulin_glucose_ratio,pregnancy_rate,days_since_checkup,activity_smoker
0,0,P00826,79,male,0,124.0,71.0,30.0,124.0,23.4,0.588,Unknown,NO,2023-04-20,0,over 55,normal,prediabetic,1.000000,0.000000,618,Unknown/NO
1,1,P00108,37,male,0,153.0,85.0,24.0,42.0,33.9,0.192,Medium,YES,2024-05-03,1,35-55,obese,diabetic,0.274510,0.000000,239,Medium/YES
2,2,P00517,39,female,0,142.0,68.0,22.0,159.0,26.9,0.777,Low,UNKNOWN,2023-07-08,0,35-55,over weight,diabetic,1.119718,0.000000,539,Low/UNKNOWN
3,3,P00687,68,female,7,121.0,88.0,26.0,124.0,29.0,1.217,Unknown,YES,2024-04-14,1,over 55,over weight,prediabetic,1.024793,0.102941,258,Unknown/YES
4,4,P00927,75,female,0,107.0,84.0,31.0,101.0,21.2,0.278,Low,NO,2024-01-27,0,over 55,normal,prediabetic,0.943925,0.000000,336,Low/NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
945,945,P00338,64,male,2,117.0,81.0,34.0,124.0,40.3,0.641,Medium,YES,2023-11-17,0,over 55,obese,prediabetic,1.059829,0.031250,407,Medium/YES
946,946,P00092,71,female,0,140.0,64.0,37.0,124.0,33.4,0.116,Low,YES,2023-05-20,1,over 55,obese,diabetic,0.885714,0.000000,588,Low/YES
947,947,P00081,40,female,0,130.0,73.0,26.0,258.0,20.1,0.709,Low,NO,2023-06-07,0,35-55,normal,diabetic,1.984615,0.000000,570,Low/NO
948,948,P00704,52,female,4,123.0,76.0,22.0,189.0,38.9,0.540,High,NO,2024-06-05,0,35-55,obese,prediabetic,1.536585,0.076923,206,High/NO


# Write the model card

In [ ]:
df.columns

Index(['Unnamed: 0', 'patient_id', 'age', 'gender', 'pregnancies', 'glucose',
       'blood_pressure', 'skin_thickness', 'insulin', 'bmi',
       'diabetes_pedigree', 'physical_activity', 'smoker', 'last_checkup_date',
       'outcome', 'age band', 'bmi category', 'glucose category',
       'insulin_glucose_ratio', 'pregnancy_rate', 'days_since_checkup',
       'activity_smoker'],
      dtype='str')

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cvres = cross_validate(final_model, x_train, y_train, cv=cv, scoring="recall")
cv_mean, cv_std = cvres["test_score"].mean(), cvres["test_score"].std()
print(f"cross validated recall: {cv_mean:.3f} +/- {cv_std:.3f}   (Week 7 record: 0.739 +/- 0.041)")

rec=[]
for col in ["age band","bmi category","glucose category"]:
    for v, s in test_df.groupby(col):
        pos = int((s["outcome"] == 1).sum())
        if pos>= 4:
            hit = int(((s["outcome"] == 1) & (s["flagged"] == 1)).sum())
            rec.append((col, v, pos, round(hit / pos, 2)))
worst = min(rec, key=lambda r: r[3])
model_card = pd.DataFrame([
    ("purpose",            "Flag patients for diabetes follow-up testing"),
    ("model",              f"Random Forest, max_dept={final_model.max_depth},"
                           f"min_samples_leaf = {final_model.min_samples_leaf}, class_weight = balanced"),
    ("Trained on",         f"{len(x_train)} patients, {x.shape[1]} features after encoding"),
    ("Test performance",   f"find{tp}of {tp + fn} diabetic patients;"
                           f"{fp} healthy patients called back; falags {pred.mean()*100:.1f} per 100"),
    ("stability",          f"cross validated recall {cv_mean:.3f} +/- {cv_std:.3f}"),
    ("Week for",           f"{worst[0]} = {worst [1]} (recall {worst[3]},{worst[2]} diabetic patients)"),
    ("use only",           "as a screening aid for a doctor, never as a diagnosis"),
    ("Review date",   "every 3 months, or sooner if the clinic's patient mix changes"),
], columns=["field","value"])
print()
print(model_card.to_string(index=False,max_colwidth=120))
model_card.to_csv(r"C:\Users\ADMIN\OneDrive\Desktop\5thsem_OJT\project1_diabetes\reports\figures\dilna_p1w8_model_card.csv",index=False)
          




cross validated recall: 0.739 +/- 0.041   (Week 7 record: 0.739 +/- 0.041)

           field                                                                              value
         purpose                                       Flag patients for diabetes follow-up testing
           model            Random Forest, max_dept=3,min_samples_leaf = 1, class_weight = balanced
      Trained on                                           760 patients, 13 features after encoding
Test performance find41of 55 diabetic patients;57 healthy patients called back; falags 51.6 per 100
       stability                                             cross validated recall 0.739 +/- 0.041
        Week for                         glucose category = normal (recall 0.0,4 diabetic patients)
        use only                              as a screening aid for a doctor, never as a diagnosis
     Review date                      every 3 months, or sooner if the clinic's patient mix changes
